# CodeGen — Group 45

## Step 7b — Instruct-model compiler-feedback repair

**Why this variant.** Step 7 fed each compile failure back to the *base*
Qwen2.5-Coder-1.5B with the rustc error and asked for a fix. It helped only a
little — 37.8% -> 39.7% standalone, 44.9% -> 46.2% stacked on the cascade — because
a **base completion model is not trained to act on an instruction like "fix this
error"**: 28 of the 34 compile errors survived untouched.

This notebook changes exactly one thing: the repair step now uses
**Qwen2.5-Coder-1.5B-Instruct**, the chat-tuned sibling (the same model the demo
uses to draft Python). Instruct models *are* trained to read an error and return a
correction, so this directly targets the 28 the base model ignored. Everything else
is identical to Step 7 — same harness, same Step 6 baseline, same crash-safe runner,
same compile-only gating — so the two rows are a clean base-vs-instruct comparison
of the *same* repair idea.

**Still honest, still safe.** Only compile errors are retried, so passes and
run-fails cannot regress. The only signal used is rustc's stderr. The repair model
is a separate step in the pipeline; the subject model whose Rust we fix is unchanged.

| Same harness | Score |
|---|---|
| Qwen-1.5B vanilla (Step 5) | 37.8% |
| compile-guided cascade (Step 6) | 44.9% |
| base-model repair, standalone (Step 7) | 39.7% |
| base-model repair + cascade (Step 7) | 46.2% |
| **Instruct repair (this step)** | **measured below** |

House rules apply. Sections 0-5 are the Step 5/6 harness unchanged; the repair model
and the chat repair prompt are the only new pieces (Sections 6-7).

## 0. Colab setup — Drive + Hugging Face token (run this first)

Everything we produce (benchmark file, model copy, eval results) lives in Drive at
`MyDrive/CodeGen_Group45`, so a crashed or recycled Colab session never loses work.

**One-time setup:** add a Colab secret (key icon in the left sidebar) named `HF_TOKEN`
containing a Hugging Face **read** token, and switch **Notebook access** ON for it.
Unauthenticated downloads from Colab are exactly what stalls / 403s (July 2026).

In [1]:
import os

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
DRIVE_ROOT = None

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/CodeGen_Group45"
    for sub in ("data", "models", "eval"):
        os.makedirs(os.path.join(DRIVE_ROOT, sub), exist_ok=True)

    # HF auth BEFORE anything talks to the Hub. Colab secret: HF_TOKEN, Notebook access ON.
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("HF token loaded from Colab secret")
    except Exception as e:
        print(f"WARNING: could not read the HF_TOKEN secret ({type(e).__name__}). "
              "Hub downloads may stall or 403 — add the secret and enable Notebook access.")
else:
    print("Not on Colab — skipping Drive; the benchmark loads from the repo's data/ folder.")

# Escape hatch only — leave False. With an upgraded hf_xet + auth, the Xet backend is the
# path that works from Colab; the non-Xet fallback was 403ing server-side (July 2026).
DISABLE_XET = False
if DISABLE_XET:
    os.environ["HF_HUB_DISABLE_XET"] = "1"

print("DRIVE_ROOT =", DRIVE_ROOT)

Mounted at /content/drive
HF token loaded from Colab secret
DRIVE_ROOT = /content/drive/MyDrive/CodeGen_Group45


## 1. Install the Rust toolchain
This gives us `rustc` (the Rust compiler). Takes ~1 minute.

In [2]:
# Install Rust (non-interactive)
!curl https://sh.rustup.rs -sSf | sh -s -- -y -q

# Make rustc/cargo visible to this notebook
import os
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

# Verify
!rustc --version
!cargo --version

warn: It looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Rustup will install the default toolchain as specified in the settings file,
warn: instead of the one inferred from the default host triple.

  stable-x86_64-unknown-linux-gnu installed - rustc 1.97.1 (8bab26f4f 2026-07-14)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source
the corresponding env file under $HOME/.cargo.

This is usually done by running one of the following (note the leading DOT):
. "$HOME/.cargo/env"            # For sh/bash/zsh/ash/dash/pdksh
source "$HOME/.cargo/env.fish"  # For fish
source "~/.cargo/env.nu"  # For nushell
source "$HOME/.cargo/env.tcsh"  # For tcsh
. "$HOME/.cargo/env.ps1"        # For pwsh
source "$HOME/.cargo/env.xsh"   # For xonsh
rustc 1.97.1 (8bab26

## 2. Install Python dependencies

Only `huggingface_hub` + its `hf_xet` download backend — and we **upgrade** them, because
Colab's preinstalled `hf_xet` is exactly what stalled our model downloads.

**Deliberately NOT installed: `datasets`.** `pip install -U datasets` drags a newer pyarrow
over Colab's preinstalled one and crashes the runtime (`IpcReadOptions size changed`).
This notebook never imports `datasets` at all — the benchmark is a plain JSONL (Section 3).

In [3]:
# Upgrade the Hub client + Xet backend BEFORE anything imports huggingface_hub.
# Do NOT add `datasets` or `torch` here (see the markdown above).
!pip install -q -U huggingface_hub hf_xet
print("done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.9/774.9 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 119.2 MB/s eta 0:00:00
done


## 3. Load the MultiPL-E Rust problems
`humaneval-rs` = 156 classic coding problems, translated into Rust, **with unit tests**.
Each problem has:
- **prompt** — the function signature + a doc comment (ends with an open `{`)
- **tests** — a `fn main()` full of `assert_eq!` checks (starts with the closing `}`)

So a complete program is simply: **prompt + the model's body + tests**.

We keep the benchmark as a plain JSONL file (repo: `data/humaneval_rs.jsonl`, Drive:
`CodeGen_Group45/data/humaneval_rs.jsonl`) and read it with stdlib `json` — no `datasets`
library, no Hub download, nothing to flake. `ds` is a plain list of dicts.

In [4]:
import json, os

def load_benchmark():
    candidates = []
    if DRIVE_ROOT:
        candidates.append(os.path.join(DRIVE_ROOT, "data", "humaneval_rs.jsonl"))
    candidates += ["data/humaneval_rs.jsonl", "../data/humaneval_rs.jsonl"]  # repo checkout
    for path in candidates:
        if os.path.exists(path):
            with open(path) as f:
                problems = [json.loads(line) for line in f if line.strip()]
            print(f"Loaded {len(problems)} problems from cache: {path}")
            return problems

    # Last resort (no Hub involved): hand-upload the repo's data/humaneval_rs.jsonl,
    # then stash it on Drive so this never happens again.
    if IN_COLAB:
        from google.colab import files
        print("Benchmark not found on Drive. Upload data/humaneval_rs.jsonl from the repo:")
        uploaded = files.upload()
        raw = next(iter(uploaded.values()))
        problems = [json.loads(line) for line in raw.decode().splitlines() if line.strip()]
        if DRIVE_ROOT:
            dest = os.path.join(DRIVE_ROOT, "data", "humaneval_rs.jsonl")
            with open(dest, "wb") as f:
                f.write(raw)
            print("Cached to Drive:", dest)
        return problems
    raise FileNotFoundError("humaneval_rs.jsonl not found — expected in the repo's data/ "
                            "folder or on Drive under CodeGen_Group45/data/.")

ds = load_benchmark()
assert len(ds) == 156, f"expected 156 problems, got {len(ds)}"
assert all(k in ds[0] for k in ("name", "prompt", "tests", "stop_tokens"))

# Look at one problem so the format is concrete
ex = ds[0]
print("\n===== PROMPT (given) =====\n", ex["prompt"])
print("===== TESTS (given) =====\n", ex["tests"])
print("===== stop tokens =====", ex["stop_tokens"])

Loaded 156 problems from cache: /content/drive/MyDrive/CodeGen_Group45/data/humaneval_rs.jsonl

===== PROMPT (given) =====
 /// Check if in given vector of numbers, are any two numbers closer to each other than
/// given threshold.
/// >>> has_close_elements(vec![1.0, 2.0, 3.0], 0.5)
/// false
/// >>> has_close_elements(vec![1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
/// true
fn has_close_elements(numbers: Vec<f64>, threshold: f64) -> bool {

===== TESTS (given) =====
 }

fn main() {
    let candidate = has_close_elements;
    assert_eq!(candidate(vec![1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3), true);
    assert_eq!(candidate(vec![1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.05), false);
    assert_eq!(candidate(vec![1.0, 2.0, 5.9, 4.0, 5.0], 0.95), true);
    assert_eq!(candidate(vec![1.0, 2.0, 5.9, 4.0, 5.0], 0.8), false);
    assert_eq!(candidate(vec![1.0, 2.0, 3.0, 4.0, 5.0, 2.0], 0.1), true);
    assert_eq!(candidate(vec![1.1, 2.2, 3.1, 4.1, 5.1], 1.0), true);
    assert_eq!(candidate(vec![1.1, 2.2, 3.1, 

## 4. The harness function
This is the heart of Step 1. It glues the three parts into one `main.rs`, compiles it,
runs it, and returns one of: `pass`, `compile_error`, `run_fail`, `compile_timeout`, `run_timeout`.

In [5]:
import subprocess, tempfile, os

def evaluate_one(prompt, completion, tests, compile_timeout=60, run_timeout=10):
    """Assemble prompt+completion+tests into a Rust program, compile and run it."""
    program = prompt + completion + tests
    with tempfile.TemporaryDirectory() as wd:
        src  = os.path.join(wd, "main.rs")
        binp = os.path.join(wd, "prog")
        with open(src, "w") as f:
            f.write(program)

        # 1) compile
        try:
            c = subprocess.run(["rustc", src, "-o", binp],
                               capture_output=True, text=True, timeout=compile_timeout)
        except subprocess.TimeoutExpired:
            return "compile_timeout"
        if c.returncode != 0:
            return "compile_error"          # didn't even build

        # 2) run against the tests
        try:
            r = subprocess.run([binp], capture_output=True, text=True, timeout=run_timeout)
        except subprocess.TimeoutExpired:
            return "run_timeout"             # probably an infinite loop
        return "pass" if r.returncode == 0 else "run_fail"

print("harness ready")

harness ready


## 5. We self-test the harness (most important step)

---


Before we trust the harness, we prove it gives the right verdict on code we already know is
correct / wrong / broken. If these three checks don't come out as we expect, the bug is in our
**harness**, not in any model.

In [6]:
ex = ds[0]   # HumanEval_0: has_close_elements(numbers: Vec<f64>, threshold: f64) -> bool

# (a) a CORRECT body  -> should PASS
correct_body = """
    for i in 0..numbers.len() {
        for j in 0..numbers.len() {
            if i != j && (numbers[i] - numbers[j]).abs() < threshold {
                return true;
            }
        }
    }
    return false;
"""

# (b) a WRONG body (compiles, but fails the tests) -> should RUN_FAIL
wrong_body = "\n    return false;\n"

# (c) a BROKEN body (does not compile) -> should COMPILE_ERROR
broken_body = "\n    return this_is_not_defined;\n"

print("correct ->", evaluate_one(ex["prompt"], correct_body, ex["tests"]))
print("wrong   ->", evaluate_one(ex["prompt"], wrong_body,   ex["tests"]))
print("broken  ->", evaluate_one(ex["prompt"], broken_body,  ex["tests"]))

assert evaluate_one(ex["prompt"], correct_body, ex["tests"]) == "pass"
assert evaluate_one(ex["prompt"], wrong_body,   ex["tests"]) == "run_fail"
assert evaluate_one(ex["prompt"], broken_body,  ex["tests"]) == "compile_error"
print("\n Harness works correctly — it can tell good Rust from bad.")

correct -> pass
wrong   -> run_fail
broken  -> compile_error

 Harness works correctly — it can tell good Rust from bad.


## 6. The repair model — Qwen2.5-Coder-1.5B-Instruct (fp16)

We do **not** load the base model here at all: the baseline to repair comes straight
from Step 6's K=0 file (Section 8), and the cascade bodies (Section 12) come from
Step 6 too. The only weights this notebook needs are the **Instruct** model that
performs the repair.

Same acquisition ladder as Steps 5/6, pointed at the Instruct checkpoint: **Drive
copy** (trusted only with the `_SAVED_OK` marker) -> **ModelScope** (primary — HF
stalled from Colab in July 2026) -> **HF Hub** (last resort, killable subprocess).
After the first run the fp16 copy is stashed on Drive, so later sessions skip the
hub entirely.

In [7]:
# Do NOT add `torch` (Colab's preinstalled torch already matches its CUDA stack)
# and do NOT add `datasets` (see Section 2).
!pip install -q -U transformers accelerate
print("done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 83.3 MB/s eta 0:00:00
done


In [8]:
import os, shutil, subprocess, sys

RMODEL_ID  = "Qwen/Qwen2.5-Coder-1.5B-Instruct"   # the repair model (chat-tuned)
RMARKER    = "_SAVED_OK"
DRIVE_RMODEL_DIR = os.path.join(DRIVE_ROOT, "models", "qwen25coder-1p5b-instruct") if DRIVE_ROOT else None
LOCAL_RDIR = "/content/qwen25coder-1p5b-instruct"

def _ms_download(model_id):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "modelscope"], check=True)
    from modelscope import snapshot_download
    return snapshot_download(model_id)

def _hf_download(model_id):
    code = f"from huggingface_hub import snapshot_download; snapshot_download('{model_id}')"
    for attempt in (1, 2):
        try:
            subprocess.run([sys.executable, "-c", code], check=True, timeout=900)
            from huggingface_hub import snapshot_download
            return snapshot_download(model_id, local_files_only=True)
        except subprocess.TimeoutExpired:
            print(f"HF Hub attempt {attempt}: no finish within 15 min (stalled) — killed")
        except subprocess.CalledProcessError:
            print(f"HF Hub attempt {attempt}: download process errored")
    raise RuntimeError("All hubs failed for " + model_id + " (Drive empty, ModelScope failed, "
                       "HF stalled/errored). Download it elsewhere and upload to Drive under "
                       "models/qwen25coder-1p5b-instruct with an empty _SAVED_OK file.")

def fetch_repair_model_dir():
    if DRIVE_RMODEL_DIR and os.path.exists(os.path.join(DRIVE_RMODEL_DIR, RMARKER)):
        if not os.path.exists(os.path.join(LOCAL_RDIR, RMARKER)):
            print("Repair model on Drive — copying to local disk (one-time per session)...")
            shutil.copytree(DRIVE_RMODEL_DIR, LOCAL_RDIR, dirs_exist_ok=True)
        print("Using the Drive copy of the repair model")
        return LOCAL_RDIR
    try:
        path = _ms_download(RMODEL_ID)
        print("Downloaded repair model from ModelScope")
        return path
    except Exception as e:
        print(f"ModelScope failed: {type(e).__name__}: {e}")
    path = _hf_download(RMODEL_ID)
    print("Downloaded repair model from the Hugging Face Hub")
    return path

repair_model_dir = fetch_repair_model_dir()
print("repair model files at:", repair_model_dir)

2026-07-29 07:41:20,627 | INFO    | modelscope_hub.download | Downloading 11 files from Qwen/Qwen2.5-Coder-1.5B-Instruct@master


Downloading:   0%|          | 0/11 [00:00<?, ?file/s]

config.json: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

README.md:   0%|          | 0.00/5.26k [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Downloaded repair model from ModelScope
repair model files at: /root/.cache/modelscope/models/Qwen--Qwen2.5-Coder-1.5B-Instruct/snapshots/master


In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), "No GPU — Runtime -> Change runtime type -> T4 GPU, then rerun."

itok = AutoTokenizer.from_pretrained(repair_model_dir)
imodel = AutoModelForCausalLM.from_pretrained(repair_model_dir, dtype=torch.float16).to("cuda")  # T4 has no bf16
imodel.eval()
print("repair (Instruct) model loaded on", imodel.device)

# One-time: stash an fp16 copy on Drive so no future session needs a hub again.
if DRIVE_RMODEL_DIR and not os.path.exists(os.path.join(DRIVE_RMODEL_DIR, RMARKER)):
    print("Saving fp16 repair-model copy to Drive (one-time, ~3 GB, a few minutes)...")
    imodel.save_pretrained(DRIVE_RMODEL_DIR)
    itok.save_pretrained(DRIVE_RMODEL_DIR)
    with open(os.path.join(DRIVE_RMODEL_DIR, RMARKER), "w") as f:
        f.write("ok\n")
    print("Saved to", DRIVE_RMODEL_DIR)

def trim_to_body(text):
    # Cut at the brace that closes the function, IGNORING braces inside strings/chars/comments.
    depth = 1
    i, n = 0, len(text)
    in_str = in_char = in_line = in_block = False
    while i < n:
        ch = text[i]
        nxt = text[i+1] if i+1 < n else ""
        if in_line:
            if ch == "\n": in_line = False
            i += 1; continue
        if in_block:
            if ch == "*" and nxt == "/": in_block = False; i += 2; continue
            i += 1; continue
        if in_str:
            if ch == "\\": i += 2; continue
            if ch == '"': in_str = False
            i += 1; continue
        if in_char:
            if ch == "\\": i += 2; continue
            if ch == "'": in_char = False
            i += 1; continue
        if ch == "/" and nxt == "/": in_line = True; i += 2; continue
        if ch == "/" and nxt == "*": in_block = True; i += 2; continue
        if ch == '"': in_str = True; i += 1; continue
        if ch == "'":
            if nxt == "\\" or (i+2 < n and text[i+2] == "'"): in_char = True
            i += 1; continue
        if ch == "{": depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0: return text[:i]
        i += 1
    return text

print("trim_to_body ready")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

repair (Instruct) model loaded on cuda:0
Saving fp16 repair-model copy to Drive (one-time, ~3 GB, a few minutes)...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/drive/MyDrive/CodeGen_Group45/models/qwen25coder-1p5b-instruct
trim_to_body ready


## 7. The chat repair pipeline

Three pieces:

1. **`compile_message`** — as in Step 7, a sibling of `evaluate_one` that returns
   rustc's stderr on failure (`evaluate_one` itself stays byte-identical for
   comparability).
2. **`build_repair_prompt`** — now a **chat** turn: a system message asking for the
   corrected function in a ```rust block with the same signature, and a user message
   carrying the broken function and the compiler error. Rendered through the
   Instruct tokenizer's chat template.
3. **body extraction** — the Instruct model replies with a *whole function* (often
   fenced), not a bare completion, so we pull the code out of the fence and keep only
   the body between the outermost braces. That body plugs into the ORIGINAL signature,
   so the graded program is still `prompt + new_body + tests` — identical grading to
   every other step.

In [10]:
import subprocess, tempfile, os, re

def compile_message(prompt, completion, tests, compile_timeout=60):
    """Compile prompt+completion+tests; return '' if it builds, else rustc's stderr.
    evaluate_one stays byte-identical across notebooks; only this helper reads stderr."""
    program = prompt + completion + tests
    with tempfile.TemporaryDirectory() as wd:
        src  = os.path.join(wd, "main.rs")
        binp = os.path.join(wd, "prog")
        with open(src, "w") as f:
            f.write(program)
        try:
            c = subprocess.run(["rustc", src, "-o", binp],
                               capture_output=True, text=True, timeout=compile_timeout)
        except subprocess.TimeoutExpired:
            return "error: compilation timed out"
        return "" if c.returncode == 0 else c.stderr

MAX_ERR_CHARS = 1200   # rustc can print a screenful; keep the prompt tight for the T4

SYSTEM_MSG = ("You are an expert Rust programmer. You are given a Rust function that "
              "fails to compile together with the exact compiler error. Reply with ONLY "
              "the corrected, complete function inside a ```rust code block, keeping the "
              "same function signature. Do not add a main function, tests, imports, or any "
              "explanation.")

def build_repair_prompt(ex, broken_body, error_text):
    """Chat repair turn -> the tokenizer's templated string. The graded program is
    still prompt + new_body + tests; we extract the body from the reply below."""
    broken_fn = ex["prompt"] + broken_body
    err = error_text.strip()
    if len(err) > MAX_ERR_CHARS:
        err = err[:MAX_ERR_CHARS] + "\n... (truncated)"
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content":
            f"```rust\n{broken_fn}\n```\n\nThe Rust compiler reported:\n{err}\n\n"
            "Return the corrected function."},
    ]
    return itok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def _extract_body(generated):
    """Pull the corrected body out of the Instruct reply: prefer a fenced ```rust
    block, then if it is a full function take the body between its outermost braces,
    else treat the text as already a body."""
    m = re.search(r"```(?:rust)?[ \t]*\r?\n?(.*?)```", generated, re.S)
    code = (m.group(1) if m else generated).strip("\n")
    if re.search(r"\bfn\s+\w+\s*\(", code):        # a whole function -> take its body
        i = code.find("{")
        if i >= 0:
            return trim_to_body(code[i+1:])
    return trim_to_body(code)                       # already just a body

def repair_completion(prompt_text, max_new_tokens=512):
    inputs = itok(prompt_text, return_tensors="pt").to(imodel.device)
    with torch.inference_mode():
        out = imodel.generate(**inputs, max_new_tokens=max_new_tokens,
                              do_sample=False, pad_token_id=itok.eos_token_id)
    gen = itok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return _extract_body(gen)

print("instruct repair pipeline ready")

instruct repair pipeline ready


### 7b. What the model actually sees

We build one chat repair prompt by hand — a deliberately broken body for problem 0
that calls a helper that does not exist — and print the templated string the Instruct
model will complete. No GPU yet; this is a pure check of the chat-prompt assembly.

In [11]:
ex = ds[0]
# Same "phantom helper" bucket the baseline hits eight times.
broken = "\n    return is_close(numbers, threshold);\n"
msg = compile_message(ex["prompt"], broken, ex["tests"])
print("rustc reported (first lines):")
print("\n".join(msg.splitlines()[:5]))
print("=" * 70)
print("Chat prompt sent to the Instruct model:\n")
print(build_repair_prompt(ex, broken, msg))

rustc reported (first lines):
error[E0425]: cannot find function `is_close` in this scope
 --> /tmp/tmpv47obw5c/main.rs:9:12
  |
9 |     return is_close(numbers, threshold);
  |            ^^^^^^^^ not found in this scope
Chat prompt sent to the Instruct model:

<|im_start|>system
You are an expert Rust programmer. You are given a Rust function that fails to compile together with the exact compiler error. Reply with ONLY the corrected, complete function inside a ```rust code block, keeping the same function signature. Do not add a main function, tests, imports, or any explanation.<|im_end|>
<|im_start|>user
```rust
/// Check if in given vector of numbers, are any two numbers closer to each other than
/// given threshold.
/// >>> has_close_elements(vec![1.0, 2.0, 3.0], 0.5)
/// false
/// >>> has_close_elements(vec![1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
/// true
fn has_close_elements(numbers: Vec<f64>, threshold: f64) -> bool {

    return is_close(numbers, threshold);

```

The Rust compi

## 8. The baseline to repair

The baseline is Step 6's vanilla K=0 completions (the 37.8%). We reuse them verbatim,
so repair starts from the identical baseline and this notebook needs no base-model
GPU time. This notebook therefore **requires** `step6_qwen_rag_k0.jsonl` on Drive — it
repairs Step 6's output rather than regenerating it.

In [12]:
import json, os, time
from collections import Counter

EVAL_DIR = os.path.join(DRIVE_ROOT, "eval") if DRIVE_ROOT else "."

def load_step6_baseline():
    path = os.path.join(EVAL_DIR, "step6_qwen_rag_k0.jsonl")
    if not os.path.exists(path):
        return None
    base = {}
    with open(path) as f:
        for line in f:
            r = json.loads(line)
            base[r["name"]] = {"status": r["status"], "body": r["body"]}
    return base if len(base) == len(ds) else None

baseline = load_step6_baseline()
if baseline is None:
    raise FileNotFoundError(
        "step6_qwen_rag_k0.jsonl (156 rows) not found in Drive eval/. This notebook "
        "repairs the Step 6 baseline, so run Step 6 first (or copy that file into eval/).")
print("Baseline = Step 6 K=0 completions (the vanilla 37.8%).")
bc = Counter(v["status"] for v in baseline.values())
print(f"baseline: pass {100*bc['pass']/len(ds):.1f}%  {dict(bc)}")
print(f"compile errors to attempt repair on: {bc['compile_error']}")

Baseline = Step 6 K=0 completions (the vanilla 37.8%).
baseline: pass 37.8%  {'pass': 59, 'run_fail': 62, 'compile_error': 34, 'run_timeout': 1}
compile errors to attempt repair on: 34


## 9. The crash-safe repair runner (+ smoke test)

The same runner as Step 7 — only compile errors enter the loop, everything else is
copied through, results stream to Drive and resume — writing to its own
`step7b_instruct_repair_*` files so the Instruct results never mix with Step 7's
base-model run.

Smoke test first (house rule): repair the first three baseline compile errors at one
round, on a throwaway file, and confirm the loop runs and returns valid verdicts.

In [13]:
REPAIR_ROUNDS = 2   # compiler-feedback attempts per compile error

def run_repair(start, max_rounds=REPAIR_ROUNDS, names=None, tag=""):
    """Walk the problems; only compile errors enter the repair loop. `start` maps
    name -> {status, body}. Streams to Drive and resumes. Own step7b file prefix."""
    path = os.path.join(EVAL_DIR, f"step7b_instruct_repair_r{max_rounds}{tag}.jsonl")
    done = {}
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                r = json.loads(line)
                done[r["name"]] = r
    targets = [ex for ex in ds if (names is None or ex["name"] in names)]
    todo = [ex for ex in targets if ex["name"] not in done]
    print(f"instruct-repair r{max_rounds}{tag}: {len(done)} done, {len(todo)} to go -> {path}")
    t0 = time.time()
    with open(path, "a") as out:
        for ex in todo:
            name = ex["name"]
            status = start[name]["status"]
            body = start[name]["body"]
            trail = [status]
            rounds = 0
            while status == "compile_error" and rounds < max_rounds:
                err = compile_message(ex["prompt"], body, ex["tests"])
                body = repair_completion(build_repair_prompt(ex, body, err))
                status = evaluate_one(ex["prompt"], body, ex["tests"])
                trail.append(status)
                rounds += 1
            rec = {"name": name, "baseline": trail[0], "status": status,
                   "rounds": rounds, "trail": trail, "body": body}
            out.write(json.dumps(rec) + "\n")
            out.flush()
            done[name] = rec
            if trail[0] == "compile_error":
                print(f"[{len(done):3d}/{len(targets)}] {name[:36]:36s} "
                      f"{trail[0]:13s} -> {status:13s} ({rounds}r, {time.time()-t0:4.0f}s)")
    return done

# --- smoke test: repair the first 3 baseline compile errors, one round, throwaway ---
ce_names = [ex["name"] for ex in ds if baseline[ex["name"]]["status"] == "compile_error"]
smoke = run_repair(baseline, max_rounds=1, names=ce_names[:3], tag="_smoke")
os.remove(os.path.join(EVAL_DIR, "step7b_instruct_repair_r1_smoke.jsonl"))
assert all(r["trail"][0] == "compile_error" for r in smoke.values()), "smoke picked wrong problems"
assert all(len(r["trail"]) == 2 for r in smoke.values()), "each smoke problem needs exactly one repair round"
print("\nsmoke OK — instruct-repair loop runs and returns valid verdicts; full run is safe")

instruct-repair r1_smoke: 0 done, 3 to go -> /content/drive/MyDrive/CodeGen_Group45/eval/step7b_instruct_repair_r1_smoke.jsonl
[  1/3] HumanEval_4_mean_absolute_deviation  compile_error -> pass          (1r,    9s)
[  2/3] HumanEval_10_make_palindrome         compile_error -> run_fail      (1r,   20s)
[  3/3] HumanEval_12_longest                 compile_error -> compile_error (1r,   29s)

smoke OK — instruct-repair loop runs and returns valid verdicts; full run is safe


## 10. The full repair run

Up to two chat repair rounds per compile error: fix, recompile, and if it still
fails, show the model the *new* error and let it try once more. Greedy decoding, so
the run is deterministic and re-runnable.

In [14]:
repaired = run_repair(baseline, max_rounds=REPAIR_ROUNDS)
rc = Counter(r["status"] for r in repaired.values())
print(f"\nafter Instruct repair (up to {REPAIR_ROUNDS} rounds): "
      f"pass {100*rc['pass']/len(ds):.1f}%  {dict(rc)}")

instruct-repair r2: 0 done, 156 to go -> /content/drive/MyDrive/CodeGen_Group45/eval/step7b_instruct_repair_r2.jsonl
[  5/156] HumanEval_4_mean_absolute_deviation  compile_error -> pass          (1r,    8s)
[ 11/156] HumanEval_10_make_palindrome         compile_error -> run_fail      (1r,   33s)
[ 13/156] HumanEval_12_longest                 compile_error -> compile_error (2r,   66s)
[ 18/156] HumanEval_17_parse_music             compile_error -> compile_error (2r,   90s)
[ 22/156] HumanEval_21_rescale_to_unit         compile_error -> compile_error (2r,  102s)
[ 37/156] HumanEval_39_prime_fib               compile_error -> compile_error (2r,  125s)
[ 42/156] HumanEval_44_change_base             compile_error -> compile_error (2r,  139s)
[ 44/156] HumanEval_46_fib4                    compile_error -> compile_error (2r,  163s)
[ 59/156] HumanEval_62_derivative              compile_error -> pass          (1r,  170s)
[ 64/156] HumanEval_67_fruit_distribution      compile_error -> run_fail 

## 11. Results — Instruct vs base-model repair

The same two views as Step 7, with the Step 7 base-model numbers printed alongside so
the base-vs-instruct comparison is right there: pass@k as rounds accumulate, and what
became of the 34 compile errors (recovered, compiled-but-wrong, or still broken).

In [15]:
print("Reference (Step 7, BASE-model repair): 39.7% standalone | 46.2% cascade+repair")
print(f"baseline:                {bc['pass']}/{len(ds)} = {100*bc['pass']/len(ds):.1f}%")
for cutoff in range(1, REPAIR_ROUNDS + 1):
    passes = sum(r["trail"][min(cutoff, len(r["trail"]) - 1)] == "pass" for r in repaired.values())
    print(f"+ instruct repair round {cutoff}: {passes}/{len(ds)} = {100*passes/len(ds):.1f}%")

ce = [r for r in repaired.values() if r["baseline"] == "compile_error"]
fixed          = [r for r in ce if r["status"] == "pass"]
compiled_wrong = [r for r in ce if r["status"] == "run_fail"]
still_ce       = [r for r in ce if r["status"] == "compile_error"]
print(f"\nof the {len(ce)} baseline compile errors:")
print(f"  recovered to PASS        : {len(fixed)}   (base-model repair recovered 3)")
print(f"  now compile but RUN_FAIL : {len(compiled_wrong)}  (compilability up, logic still wrong)")
print(f"  still COMPILE_ERROR      : {len(still_ce)}")
print(f"\nrecovered to pass: {[r['name'] for r in fixed]}")

Reference (Step 7, BASE-model repair): 39.7% standalone | 46.2% cascade+repair
baseline:                59/156 = 37.8%
+ instruct repair round 1: 63/156 = 40.4%
+ instruct repair round 2: 64/156 = 41.0%

of the 34 baseline compile errors:
  recovered to PASS        : 5   (base-model repair recovered 3)
  now compile but RUN_FAIL : 9  (compilability up, logic still wrong)
  still COMPILE_ERROR      : 20

recovered to pass: ['HumanEval_4_mean_absolute_deviation', 'HumanEval_62_derivative', 'HumanEval_74_total_match', 'HumanEval_110_exchange', 'HumanEval_115_max_fill']


## 12. Stacking with the Step 6 cascade

The headline number: run the Step 6 cascade first (K=0 -> idiom+k2 -> k4, first body
that compiles), then hand whatever still fails to compile to the Instruct repair loop.
Every gate is a compile signal, so the composed system stays inference-legal. Compare
directly against Step 7's base-model composition (46.2%).

In [16]:
def load_bodies(label):
    path = os.path.join(EVAL_DIR, f"step6_qwen_rag_{label}.jsonl")
    if not os.path.exists(path):
        return None
    d = {}
    with open(path) as f:
        for line in f:
            r = json.loads(line)
            d[r["name"]] = {"status": r["status"], "body": r["body"]}
    return d if len(d) == len(ds) else None

k0b, idi, k4b = load_bodies("k0"), load_bodies("k2_idiom"), load_bodies("k4")
if k0b and idi and k4b:
    def cascade_pick(name):
        for src in (k0b, idi, k4b):          # first body that compiles wins
            if src[name]["status"] != "compile_error":
                return {"status": src[name]["status"], "body": src[name]["body"]}
        return {"status": k4b[name]["status"], "body": k4b[name]["body"]}
    cascade_start = {ex["name"]: cascade_pick(ex["name"]) for ex in ds}
    casc_pass = sum(v["status"] == "pass" for v in cascade_start.values())
    print(f"Step 6 cascade alone:                        {casc_pass}/{len(ds)} = {100*casc_pass/len(ds):.1f}%")

    composed = run_repair(cascade_start, max_rounds=REPAIR_ROUNDS, tag="_cascade")
    cp = Counter(r["status"] for r in composed.values())
    print(f"cascade + Instruct compiler-feedback repair: {cp['pass']}/{len(ds)} = {100*cp['pass']/len(ds):.1f}%")
    print(f"  (Step 7 base-model composition was 46.2%)")
    print(f"  remaining compile errors: {cp['compile_error']}   full mix: {dict(cp)}")
else:
    print("Step 6 sweep files (k0 / k2_idiom / k4) not on Drive — skipping composition.")
    print("Run Step 6 first, or copy those step6_qwen_rag_*.jsonl files into eval/.")

Step 6 cascade alone:                        70/156 = 44.9%
instruct-repair r2_cascade: 0 done, 156 to go -> /content/drive/MyDrive/CodeGen_Group45/eval/step7b_instruct_repair_r2_cascade.jsonl
[  5/156] HumanEval_4_mean_absolute_deviation  compile_error -> pass          (1r,    7s)
[ 18/156] HumanEval_17_parse_music             compile_error -> compile_error (2r,   29s)
[ 37/156] HumanEval_39_prime_fib               compile_error -> compile_error (2r,   47s)
[ 59/156] HumanEval_62_derivative              compile_error -> pass          (1r,   54s)
[ 65/156] HumanEval_68_pluck                   compile_error -> run_fail      (1r,   58s)
[106/156] HumanEval_110_exchange               compile_error -> run_fail      (1r,   62s)
[125/156] HumanEval_130_tri                    compile_error -> run_fail      (1r,   75s)
[135/156] HumanEval_141_file_name_check        compile_error -> compile_error (2r,  117s)
[137/156] HumanEval_143_words_in_sentence      compile_error -> compile_error (2r,  145

## What this step adds

- A **base-vs-instruct** comparison of compiler-feedback repair on the identical 34
  compile errors: does a chat-tuned model actually use the error the base model
  ignored? The table in Section 11 answers it directly.
- The project's **best inference-legal number** if the composition beats 46.2% — still
  gated only on compilation, still unable to regress a pass.

If Instruct repair moves the number, it is the headline system for the report; if it
does not, that is the clean finding that these compile errors are genuine capability
limits, not prompt-format artifacts. Either way the next row is Qwen2.5-Coder-7B in
4-bit as the large-model reference for the ~61 logic errors no repair can reach, then
the CP3 comparison table.